# Data Cleaning and Preparation — Freedom in the World

**Notebook 03 of 08**

### Purpose

Turn the wide raw CSV (one row per economy × indicator, with year columns) into the **clean long-format analytical dataset** — one row per economy × indicator × year — and save it as `data/processed/freedom_in_world_long.csv`. Every cleaning decision is documented here so the analysis is reproducible.

### Scope

The raw files in `data/raw/` are **never modified**. This notebook reads them (plus the lookup tables from notebook 02) and writes only to `data/processed/`.

## Data and Inputs

| Item | Location | Role |
|---|---|---|
| Raw wide CSV | `data/raw/FH_FIW_WIDEF.csv` | The only data source |
| Economy lookup | `data/processed/lookup_tables/economy_lookup.csv` | Verified in notebook 02 |
| Indicator lookup | `data/processed/lookup_tables/indicator_lookup.csv` | Adds the Category column |

**Output:** `data/processed/freedom_in_world_long.csv` — the analytical dataset that every later notebook and the dashboard consume.

**Question this notebook answers:** *How do we go from wide (economy × indicator × year columns) to long (economy × indicator × year × score) without losing or inventing information?*

## Setup: imports and the project root

Same bootstrap as notebooks 01–02. This notebook uses `load_raw_data()` and the shared `reshape_freedom_data()` helper from `src/data_cleaning.py` — the cleaning logic lives in `src/`, the notebook demonstrates and documents it.

In [1]:
import sys
from pathlib import Path

current = Path.cwd()
while not (current / 'data' / 'raw' / 'FH_FIW_WIDEF.csv').exists():
    current = current.parent
    if current == current.parent:
        raise RuntimeError('Could not find the project root.')

if str(current) not in sys.path:
    sys.path.insert(0, str(current))

import pandas as pd

from src.data_loader import load_raw_data
from src.data_cleaning import reshape_freedom_data

# Notebook 02 outputs feed this notebook
lookup_dir = current / 'data' / 'processed' / 'lookup_tables'
countries = pd.read_csv(lookup_dir / 'economy_lookup.csv')
indicators = pd.read_csv(lookup_dir / 'indicator_lookup.csv')

df = load_raw_data()
print('raw wide CSV:', df.shape)
print('economy lookup:', countries.shape)
print('indicator lookup:', indicators.shape)

raw wide CSV: (7880, 53)
economy lookup: (197, 2)
indicator lookup: (40, 5)


### Interpretation

All inputs loaded: the raw wide frame and the two lookup tables saved by notebook 02. The lookups exist because notebook 02 ran — the notebooks are a sequential pipeline.

## 1. The cleaning plan

**Question:** what exactly does this notebook change, and why?

**Method:** every decision is stated up front; the rest of the notebook executes and verifies it.

| Decision | Rationale |
|---|---|
| Raw files in `data/raw/` are never modified | The raw data is the source of truth; cleaning outputs go to `data/processed/` |
| Economy name: `REF_AREA_LABEL` renamed to `Economy` | Matches the documented long shape `Economy \| Indicator \| Year \| Score`; notebook 02 proved the CSV names equal the metadata lookup |
| Year strings converted to integers | Year columns are read as `str` (notebook 01); years are numeric by nature |
| Scores coerced to numeric | Numeric indicators should hold numbers; non-numeric cells become NaN (missing) |
| Missing values are never imputed | Project policy — missing stays missing unless a documented analytical reason says otherwise |
| STATUS scores stay strings (`F`/`PF`/`NF`) | `FH_FIW_STATUS` is categorical (`UNIT_MEASURE = CAT`); numeric coercion would destroy it |
| Category attached from the indicator lookup | The category is derived from the CSV's own labels (notebook 02) |
| No deduplication step | Notebook 01 found zero duplicates; this notebook re-verifies after reshaping |

## 2. Verify the inputs

**Question:** do the lookup files still cover the raw CSV completely?

**Method:** re-assert that the code sets match, now that the lookups come from files.

In [2]:
raw_codes = set(df['REF_AREA'])
assert raw_codes == set(countries['REF_AREA']), 'economy lookup mismatch'

ind_codes = set(df['INDICATOR'])
assert ind_codes == set(indicators['INDICATOR']), 'indicator lookup mismatch'

print('Economy lookup covers all', len(raw_codes), 'CSV codes.')
print('Indicator lookup covers all', len(ind_codes), 'CSV codes.')

Economy lookup covers all 197 CSV codes.
Indicator lookup covers all 40 CSV codes.


### Interpretation

The saved lookup files are consistent with the raw CSV — no codes are missing in either direction. The join in the reshape step will therefore be complete.

## 3. The reshape: wide to long

**Question:** how do we melt the 14 year columns into rows?

**Method:** call the shared `reshape_freedom_data()` — it melts, converts years to int, coerces scores, keeps STATUS as strings, and attaches `Category`.

In [3]:
long = reshape_freedom_data(df, indicators=indicators)
print('Long dataset shape:', long.shape)
print('Columns:', list(long.columns))
long.head()

Long dataset shape: (110320, 9)
Columns: ['REF_AREA', 'Economy', 'INDICATOR', 'INDICATOR_LABEL', 'Category', 'UNIT_MEASURE', 'UNIT_MEASURE_LABEL', 'Year', 'Score']


,REF_AREA,Economy,INDICATOR,INDICATOR_LABEL,Category,UNIT_MEASURE,UNIT_MEASURE_LABEL,Year,Score
0,COD,"Congo, Dem. Rep.",FH_FIW_F3,Rule of Law: Is there protection from the ille...,Rule of Law,0_TO_4,0-4 scale,2013,0.0
1,MYS,Malaysia,FH_FIW_F3,Rule of Law: Is there protection from the ille...,Rule of Law,0_TO_4,0-4 scale,2013,1.0
2,TZA,Tanzania,FH_FIW_F4,"Rule of Law: Do laws, policies, and practices ...",Rule of Law,0_TO_4,0-4 scale,2013,3.0
3,TZA,Tanzania,FH_FIW_G2,Personal Autonomy And Individual Rights: Are i...,Personal Autonomy And Individual Rights,0_TO_4,0-4 scale,2013,2.0
4,BEL,Belgium,FH_FIW_G3,Personal Autonomy And Individual Rights: Do in...,Personal Autonomy And Individual Rights,0_TO_4,0-4 scale,2013,4.0


### Interpretation

Each of the 7880 wide rows became **14 long rows** — one per year — so the long dataset has **7880 × 14 = 110,320 rows** and 9 columns: codes and labels for the economy, indicator (with category and scale), `Year` and `Score`. This is the shape every later notebook works with.

## 4. Verify dtypes and years

**Question:** did the conversions actually happen?

**Method:** inspect the dtypes and the year range.

In [4]:
print('Year range:', long['Year'].min(), '-', long['Year'].max())
print()
print(long.dtypes.to_string())

Year range: 2013 - 2026

REF_AREA                 str
Economy                  str
INDICATOR                str
INDICATOR_LABEL          str
Category                 str
UNIT_MEASURE             str
UNIT_MEASURE_LABEL       str
Year                   int64
Score                 object


### Interpretation

`Year` is now `int64` (2013–2026). `Score` is `object` — deliberate: it holds floats for the 39 numeric indicators and strings for the categorical STATUS indicator. All descriptor columns are strings; `Category` was attached cleanly.

## 5. Duplicates after reshaping

**Question:** did the melt create any duplicate rows?

**Method:** count rows sharing the same economy, indicator and year.

In [5]:
dupes = long.duplicated(subset=['REF_AREA', 'INDICATOR', 'Year']).sum()
print('Duplicate (economy, indicator, year) rows:', dupes)

Duplicate (economy, indicator, year) rows: 0


### Interpretation

**Zero duplicates** — each economy × indicator × year combination appears exactly once, confirming the cleaning plan needs no deduplication step.

## 6. Missing values in the long dataset

**Question:** where does missingness sit after reshaping?

**Method:** count missing cells per column.

In [6]:
print('Missing cells per column:')
print(long.isna().sum().to_string())
print('Total missing:', long.isna().sum().sum())

Missing cells per column:
REF_AREA                 0
Economy                  0
INDICATOR                0
INDICATOR_LABEL          0
Category                 0
UNIT_MEASURE             0
UNIT_MEASURE_LABEL       0
Year                     0
Score                 2164
Total missing: 2164


### Interpretation

All **2164** missing cells sit in `Score` — exactly the year-cell missingness notebook 01 found in the raw CSV (none in 2013–2016, appearing from 2017 onward). Nothing was imputed; the numbers survived the reshape unchanged.

## 7. Unexpected score values

**Question:** do the observed scores stay within their nominal scales?

**Method:** take the min and max per `UNIT_MEASURE` across the numeric indicators (STATUS excluded — it is categorical).

In [7]:
numeric_rows = long[long['UNIT_MEASURE'] != 'CAT']
ranges = numeric_rows.groupby(['UNIT_MEASURE', 'UNIT_MEASURE_LABEL']).agg(
    observed_min=('Score', 'min'),
    observed_max=('Score', 'max'),
)
ranges

,,observed_min,observed_max
UNIT_MEASURE,UNIT_MEASURE_LABEL,,
0_TO_100,0-100 scale,-1.0,100.0
0_TO_12,0-12 scale,0.0,12.0
0_TO_16,0-16 scale,0.0,16.0
0_TO_4,0-4 scale,0.0,4.0
0_TO_40,0-40 scale,-4.0,40.0
0_TO_60,0-60 scale,1.0,60.0
1_TO_7,1-7 scale,1.0,7.0


### Interpretation

Most scales sit inside their nominal bounds, with one documented exception: `FH_FIW_PR` (`0_TO_40`) shows a **minimum of −4** — South Sudan and Sudan, as notebook 01 flagged. We keep these values: `UNIT_MEASURE` is a nominal description, not a validation bound. Also visible: the 1–7 rating scale (remember it runs opposite — 1 is most free).

## 8. The categorical STATUS column

**Question:** is the status classification intact after cleaning?

**Method:** look at the distinct values of `Score` for `FH_FIW_STATUS` rows.

In [8]:
status = long[long['INDICATOR'] == 'FH_FIW_STATUS']
print('STATUS values preserved:', sorted(status['Score'].dropna().unique()))
print('STATUS rows with a missing score:', status['Score'].isna().sum())

STATUS values preserved: ['F', 'NF', 'PF']
STATUS rows with a missing score: 10


### Interpretation

The three statuses **Free (`F`), Partly Free (`PF`), Not Free (`NF`)** survived cleaning as strings. A handful of STATUS rows are missing (10) — these are genuinely missing in the raw data and stay missing. This is the payoff of excluding the `CAT` scale from numeric coercion.

## 9. Save the analytical dataset

**Question:** where does the pipeline's clean dataset live?

**Method:** write the long dataset to `data/processed/` and prove it loads back intact.

In [9]:
out_path = current / 'data' / 'processed' / 'freedom_in_world_long.csv'
long.to_csv(out_path, index=False)
print('Saved', out_path)
print()

# Round-trip check: the saved file must load back to the same dataset
reloaded = pd.read_csv(out_path)
print('Reloaded shape:', reloaded.shape)
print('Reloaded Year dtype:', reloaded['Year'].dtype)
reloaded.head()

Saved C:\Users\OgwalJoshuaRobin\OneDrive - War Child\Desktop\Freedom\data\processed\freedom_in_world_long.csv



Reloaded shape: (110320, 9)
Reloaded Year dtype: int64


,REF_AREA,Economy,INDICATOR,INDICATOR_LABEL,Category,UNIT_MEASURE,UNIT_MEASURE_LABEL,Year,Score
0,COD,"Congo, Dem. Rep.",FH_FIW_F3,Rule of Law: Is there protection from the ille...,Rule of Law,0_TO_4,0-4 scale,2013,0.0
1,MYS,Malaysia,FH_FIW_F3,Rule of Law: Is there protection from the ille...,Rule of Law,0_TO_4,0-4 scale,2013,1.0
2,TZA,Tanzania,FH_FIW_F4,"Rule of Law: Do laws, policies, and practices ...",Rule of Law,0_TO_4,0-4 scale,2013,3.0
3,TZA,Tanzania,FH_FIW_G2,Personal Autonomy And Individual Rights: Are i...,Personal Autonomy And Individual Rights,0_TO_4,0-4 scale,2013,2.0
4,BEL,Belgium,FH_FIW_G3,Personal Autonomy And Individual Rights: Do in...,Personal Autonomy And Individual Rights,0_TO_4,0-4 scale,2013,4.0


### Interpretation

The dataset is saved as `data/processed/freedom_in_world_long.csv` and reloads cleanly: same shape (110,320 × 9) and `Year` reads back as an integer. This file is the contract for everything that follows — notebooks 04–08 and the Streamlit dashboard all consume it and never touch the raw CSV.

## Summary and next question

### What we learned

- The wide CSV was reshaped to **long format**: 110,320 rows (economy × indicator × year) with codes, labels, category, scale, `Year` and `Score`.
- Cleaning decisions, all documented: years → int, scores → numeric (STATUS excluded, kept as strings), missing **never imputed**, no deduplication needed.
- **2164 missing scores** survived unchanged; the only out-of-scale values are the documented PR = −4 cases.
- The analytical dataset now lives at `data/processed/freedom_in_world_long.csv`, verified by a round-trip read.

### Next question

*How have Freedom in the World scores changed globally between 2013 and 2026?* — that is notebook 04, global trends.